In [1]:
import os
import math
import torch
import zipfile
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

In [2]:
# 登录 Hugging Face
from huggingface_hub import login

login()
print("✅ 登录完成")

✅ 登录完成


In [ ]:
# ============================================
# 1. 配置（与 LLaDA notebook 完全一致）
# ============================================
CFG = {
    "model_name": "meta-llama/Llama-3.1-8B",
    "max_length": 2048,
    "lora_r": 8,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
    "num_epochs": 3,
    "batch_size": 1,
    "grad_accum_steps": 8,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "warmup_steps": 20,
    "log_every": 10,
    "save_every": 100,
    "output_dir": "./llama_base_medical_lora",
}

# ============================================
# 2. 加载数据（合并 gs + large-scale）
# ============================================
print("\n📂 Loading data...")

def load_from_zip(zip_path):
    records = []
    if not os.path.exists(zip_path):
        print(f"   ⚠️ File not found: {zip_path}")
        return pd.DataFrame()
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if '/fulltext/' in file and file.endswith('.txt'):
                try:
                    full_text = z.read(file).decode('utf-8')
                    summary_file = file.replace('/fulltext/', '/summaries/').replace('.txt', '_sum.txt')
                    summary_text = z.read(summary_file).decode('utf-8')
                    records.append({'Full_Text': full_text, 'Summary': summary_text})
                except:
                    continue
    return pd.DataFrame(records)

df_gs = load_from_zip('multiclinsum_gs_train_en.zip')
df_large = load_from_zip('multiclinsum_large-scale_train_en.zip')
df = pd.concat([df_gs, df_large], ignore_index=True)
print(f"✅ Total: {len(df)} samples (GS: {len(df_gs)} + Large: {len(df_large)})")

if len(df) == 0:
    raise FileNotFoundError("No data loaded! Check zip file paths.")

# 拆分训练/验证集（90/10）
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Train: {len(train_df)} | Validation: {len(eval_df)}")

# ============================================
# 3. Dataset（与 LLaDA notebook 完全一致）
# ============================================
class ClinicalSumDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.samples = []
        eos_id = tokenizer.eos_token_id
        skipped = 0

        for _, row in df.iterrows():
            # 和 LLaDA notebook 完全一样的 prompt 格式
            prompt_text = f"Summarize this clinical note: {row['Full_Text']}\nSummary: "
            response_text = row["Summary"]

            prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
            response_ids = tokenizer.encode(response_text, add_special_tokens=False)
            response_ids = response_ids + [eos_id]

            if len(prompt_ids) >= max_length:
                skipped += 1
                continue

            full_ids = (prompt_ids + response_ids)[:max_length]
            pad_len = max_length - len(full_ids)
            full_ids = full_ids + [eos_id] * pad_len

            self.samples.append({
                "input_ids": torch.tensor(full_ids, dtype=torch.long),
                "prompt_length": torch.tensor(len(prompt_ids), dtype=torch.long),
            })

        print(f"Dataset: {len(self.samples)} usable, {skipped} skipped (prompt > max_length).")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]



📂 Loading data...
✅ Total: 26494 samples (GS: 592 + Large: 25902)
Train: 23844 | Validation: 2650


In [ ]:
# ============================================
# 正确的模型加载（不修改 rope_scaling）
# ============================================

print("\n🦙 Loading model...")

# 不要修改 config，直接加载
config = AutoConfig.from_pretrained(CFG["model_name"], trust_remote_code=True)

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
pad_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Model loaded")


🦙 Loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded


In [ ]:
# ============================================
# 5. 创建 DataLoader
# ============================================
train_dataset = ClinicalSumDataset(
    train_df,
    tokenizer,
    CFG["max_length"]
)

eval_dataset = ClinicalSumDataset(
    eval_df,
    tokenizer,
    CFG["max_length"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

# ============================================
# 6. LoRA 配置
# ============================================
print("\n🔧 Applying LoRA...")

lora_config = LoraConfig(
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

total_params = sum(p.numel() for p in model.parameters())

print(
    f"Trainable: {trainable_params:,} / "
    f"{total_params:,} "
    f"({100 * trainable_params / total_params:.4f}%)"
)

# ============================================
# 7. 优化器和 Scheduler
# ============================================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"],
)

updates_per_epoch = math.ceil(
    len(train_loader) / CFG["grad_accum_steps"]
)

total_steps = updates_per_epoch * CFG["num_epochs"]

warmup_steps = CFG["warmup_steps"]

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)

    progress = (
        (step - warmup_steps)
        / max(1, total_steps - warmup_steps)
    )

    return max(
        0.0,
        0.5 * (1.0 + math.cos(math.pi * progress))
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

# ============================================
# 8. Training
# ============================================
print("\n🚀 Starting training...")
print(f"Training — {CFG['num_epochs']} epochs")
print(
    f"Effective batch size: "
    f"{CFG['batch_size'] * CFG['grad_accum_steps']}"
)
print("-" * 60)

model.train()

global_step = 0
running_loss = 0.0

optimizer.zero_grad()

for epoch in range(CFG["num_epochs"]):

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}"
    )

    for step, batch in enumerate(progress_bar):

        # ====================================
        # batch
        # ====================================
        input_ids = batch["input_ids"].to(model.device)

        prompt_lengths = batch["prompt_length"].to(
            model.device
        )

        # ====================================
        # labels
        # ====================================
        labels = input_ids.clone()

        # mask prompt
        for i, pl in enumerate(prompt_lengths):
            labels[i, :pl] = -100

        # attention mask
        attention_mask = (
            input_ids != pad_id
        ).long()

        # mask padding only
        labels[attention_mask == 0] = -100

        # ====================================
        # forward
        # ====================================
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        # ====================================
        # gradient accumulation
        # ====================================
        loss = loss / CFG["grad_accum_steps"]

        loss.backward()

        running_loss += (
            loss.item() * CFG["grad_accum_steps"]
        )

        # ====================================
        # optimizer step
        # ====================================
        if (
            (step + 1) % CFG["grad_accum_steps"] == 0
            or (step + 1) == len(train_loader)
        ):

            # gradient clipping
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad()

            global_step += 1

            # ================================
            # logging
            # ================================
            if global_step % CFG["log_every"] == 0:

                avg_loss = (
                    running_loss / CFG["log_every"]
                )

                lr_now = scheduler.get_last_lr()[0]

                print(
                    f"Epoch {epoch+1} | "
                    f"Step {global_step:5d} | "
                    f"Loss {avg_loss:.4f} | "
                    f"LR {lr_now:.2e}"
                )

                running_loss = 0.0

            # ================================
            # save checkpoint
            # ================================
            if global_step % CFG["save_every"] == 0:

                os.makedirs(
                    CFG["output_dir"],
                    exist_ok=True
                )

                checkpoint_path = os.path.join(
                    CFG["output_dir"],
                    f"checkpoint-{global_step}"
                )

                model.save_pretrained(
                    checkpoint_path
                )

                tokenizer.save_pretrained(
                    checkpoint_path
                )

                print(
                    f"Checkpoint saved -> "
                    f"{checkpoint_path}"
                )

# ============================================
# 9. Save final model
# ============================================
print("\n💾 Saving final model...")

os.makedirs(CFG["output_dir"], exist_ok=True)

model.save_pretrained(CFG["output_dir"])

tokenizer.save_pretrained(CFG["output_dir"])

print(f"✅ Saved to {CFG['output_dir']}")

Dataset: 23670 usable, 174 skipped (prompt > max_length).
Dataset: 2637 usable, 13 skipped (prompt > max_length).

🔧 Applying LoRA...
Trainable: 6,815,744 / 8,037,076,992 (0.0848%)

🚀 Starting training...
Training — 3 epochs
Effective batch size: 8
------------------------------------------------------------


Epoch 1:   0%|          | 0/23670 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARAL

Epoch 1 | Step    10 | Loss 103.7055 | LR 1.00e-04


Epoch 1:   1%|          | 160/23670 [01:20<3:18:18,  1.98it/s]

Epoch 1 | Step    20 | Loss 14.6264 | LR 2.00e-04


...

Epoch 1 | Step   110 | Loss 0.0009 | LR 2.00e-04


Epoch 1:   4%|▍         | 960/23670 [08:15<3:17:41,  1.91it/s]

Epoch 1 | Step   120 | Loss 0.0009 | LR 2.00e-04


Epoch 1:   4%|▍         | 978/23670 [08:24<3:15:08,  1.94it/s]


KeyboardInterrupt: 

In [15]:
import torch
import gc

# 1. 清空 CUDA 缓存
torch.cuda.empty_cache()
gc.collect()

0

In [6]:
# 快速测试 checkpoint-100
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 加载基础模型
base_model_name = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# 加载 LoRA
model = PeftModel.from_pretrained(base_model, "./llama_base_medical_lora/checkpoint-100")
model.eval()

# 测试
prompt = f"Summarize this clinical note: {eval_df.iloc[0]['Full_Text']}\nSummary: "
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Generated: {generated[:300]}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Generated:                                                                                                                                                                                                         
